# MaStR Data Analysis

Search the Marktstammdatenregister (BNetzA) export for plants behind redispatch cluster entries that are missing from PyPSA.

**Data:** `Gesamtdatenexport_20260328_25.2/` (UTF-16 encoded XML files)

Files searched:
- `EinheitenWind.xml` — all wind turbines
- `EinheitenSolar_*.xml` — all solar units (split across ~60 files)
- `EinheitenBiomasse.xml` — biomass units
- `EinheitenVerbrennung.xml` — combustion units (gas, coal, oil)

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import glob
import os

MASTR_DIR = "Gesamtdatenexport_20260328_25.2"

# Common fields to extract from any unit type
COMMON_FIELDS = [
    "EinheitMastrNummer",
    "NameStromerzeugungseinheit",
    "Gemeinde",
    "Gemarkung",
    "Ort",
    "Landkreis",
    "Postleitzahl",
    "Bruttoleistung",
    "Laengengrad",
    "Breitengrad",
    "Inbetriebnahmedatum",
    "EinheitBetriebsstatus",
]

def search_xml(xml_path: str, unit_tag: str, keywords: list[str], fields=COMMON_FIELDS) -> pd.DataFrame:
    """Stream-parse a MaStR XML file and return rows matching any keyword in location fields."""
    hits = []
    with open(xml_path, "rb") as f:
        for event, elem in ET.iterparse(f, events=("end",)):
            if elem.tag != unit_tag:
                continue
            row = {field: (elem.findtext(field) or "").strip() for field in fields}
            search_text = " ".join([
                row["Gemeinde"], row["Gemarkung"], row["Ort"], row["Landkreis"]
            ]).lower()
            if any(kw.lower() in search_text for kw in keywords):
                hits.append(row)
            elem.clear()
    return pd.DataFrame(hits)

def search_solar(keywords: list[str], fields=COMMON_FIELDS) -> pd.DataFrame:
    """Search all EinheitenSolar_*.xml files (split across ~60 files)."""
    solar_files = sorted(glob.glob(f"{MASTR_DIR}/EinheitenSolar_*.xml"))
    all_hits = []
    for path in solar_files:
        hits = search_xml(path, "EinheitSolar", keywords, fields)
        if not hits.empty:
            all_hits.append(hits)
    return pd.concat(all_hits, ignore_index=True) if all_hits else pd.DataFrame()

def summarise(df: pd.DataFrame, label: str) -> None:
    if df.empty:
        print(f"  {label}: NOT FOUND")
        return
    df["Bruttoleistung"] = pd.to_numeric(df["Bruttoleistung"], errors="coerce")
    print(f"  {label}: {len(df)} units, {df['Bruttoleistung'].sum():.1f} MW total")
    print(f"    Gemeinde : {df['Gemeinde'].unique().tolist()}")
    print(f"    Gemarkung: {df['Gemarkung'].unique().tolist()[:6]}")

print("Helpers loaded.")

## Süderdonn

Small municipality in Dithmarschen, Schleswig-Holstein. Wind turbines may be registered under neighboring Gemeinden.
Also searching Dithmarschen broadly and nearby towns (Lunden, Barlt, Eiderstedt area).

In [ ]:
# Broad search — Süderdonn directly + neighboring area keywords
SUDERDONN_KEYWORDS = [
    "süderdonn", "suderdonn",
    "lunden",       # neighboring municipality
    "barlt",        # neighboring municipality
    "windbergen",   # neighboring municipality
    "pahlen",       # nearby
]

print("Searching wind...")
sud_wind = search_xml(f"{MASTR_DIR}/EinheitenWind.xml", "EinheitWind", SUDERDONN_KEYWORDS)
summarise(sud_wind, "Wind")

print("Searching solar...")
sud_solar = search_solar(SUDERDONN_KEYWORDS)
summarise(sud_solar, "Solar")

print("Searching biomass...")
sud_bio = search_xml(f"{MASTR_DIR}/EinheitenBiomasse.xml", "EinheitBiomasse", SUDERDONN_KEYWORDS)
summarise(sud_bio, "Biomass")

In [ ]:
# Show full wind results for Süderdonn area
if not sud_wind.empty:
    sud_wind["Bruttoleistung"] = pd.to_numeric(sud_wind["Bruttoleistung"], errors="coerce")
    display(sud_wind[[
        "EinheitMastrNummer", "NameStromerzeugungseinheit",
        "Gemeinde", "Gemarkung", "Ort",
        "Bruttoleistung", "Laengengrad", "Breitengrad",
        "Inbetriebnahmedatum", "EinheitBetriebsstatus"
    ]].sort_values("Gemeinde"))

## Voslapp

Industrial port district (Stadtteil) of Wilhelmshaven, Lower Saxony.
Wind turbines in the harbor area may be registered under `Wilhelmshaven` as Gemeinde
with `Voslapp` appearing in Gemarkung or Ort — or under a completely different cadastral name.

In [ ]:
VOSLAPP_KEYWORDS = [
    "voslapp",
    "wilhelmshaven",   # parent municipality
    "rüstringen",      # historical district of Wilhelmshaven
]

print("Searching wind...")
vos_wind = search_xml(f"{MASTR_DIR}/EinheitenWind.xml", "EinheitWind", VOSLAPP_KEYWORDS)
summarise(vos_wind, "Wind")

print("Searching solar...")
vos_solar = search_solar(VOSLAPP_KEYWORDS)
summarise(vos_solar, "Solar")

print("Searching biomass...")
vos_bio = search_xml(f"{MASTR_DIR}/EinheitenBiomasse.xml", "EinheitBiomasse", VOSLAPP_KEYWORDS)
summarise(vos_bio, "Biomass")

print("Searching combustion (gas/oil)...")
vos_verb = search_xml(f"{MASTR_DIR}/EinheitenVerbrennung.xml", "EinheitVerbrennung", VOSLAPP_KEYWORDS)
summarise(vos_verb, "Combustion")

In [ ]:
# Drill into Gemarkung to find Voslapp specifically within Wilhelmshaven wind results
if not vos_wind.empty:
    vos_wind["Bruttoleistung"] = pd.to_numeric(vos_wind["Bruttoleistung"], errors="coerce")
    print("Gemarkung breakdown for Wilhelmshaven wind:")
    print(vos_wind.groupby("Gemarkung")["Bruttoleistung"].agg(["count", "sum"]).sort_values("sum", ascending=False))
    print()
    print("Ort breakdown:")
    print(vos_wind.groupby("Ort")["Bruttoleistung"].agg(["count", "sum"]).sort_values("sum", ascending=False))

In [ ]:
# Filter to just Voslapp entries if any exist
if not vos_wind.empty:
    voslapp_only = vos_wind[
        vos_wind["Gemarkung"].str.contains("voslapp", case=False, na=False) |
        vos_wind["Ort"].str.contains("voslapp", case=False, na=False)
    ]
    if not voslapp_only.empty:
        print(f"Found {len(voslapp_only)} turbines explicitly labelled Voslapp:")
        display(voslapp_only[[
            "EinheitMastrNummer", "NameStromerzeugungseinheit",
            "Gemeinde", "Gemarkung", "Ort",
            "Bruttoleistung", "Laengengrad", "Breitengrad",
            "Inbetriebnahmedatum"
        ]])
    else:
        print("No entries with 'Voslapp' in Gemarkung or Ort — registered under different cadastral name.")
        print("See Gemarkung breakdown above to identify the likely candidate.")

## Pleinting

Small municipality near Straubing, Lower Bavaria. Likely solar (appears as Solar in PyPSA, BAG NWAK cluster → Bayernwerk territory).

In [ ]:
PLEINTING_KEYWORDS = ["pleinting", "niederwinkling", "steinach"]  # neighboring municipalities

print("Searching solar...")
pl_solar = search_solar(PLEINTING_KEYWORDS)
summarise(pl_solar, "Solar")

print("Searching wind...")
pl_wind = search_xml(f"{MASTR_DIR}/EinheitenWind.xml", "EinheitWind", PLEINTING_KEYWORDS)
summarise(pl_wind, "Wind")

print("Searching biomass...")
pl_bio = search_xml(f"{MASTR_DIR}/EinheitenBiomasse.xml", "EinheitBiomasse", PLEINTING_KEYWORDS)
summarise(pl_bio, "Biomass")

if not pl_solar.empty:
    pl_solar["Bruttoleistung"] = pd.to_numeric(pl_solar["Bruttoleistung"], errors="coerce")
    display(pl_solar[[
        "EinheitMastrNummer", "NameStromerzeugungseinheit",
        "Gemeinde", "Gemarkung",
        "Bruttoleistung", "Laengengrad", "Breitengrad",
        "Inbetriebnahmedatum", "EinheitBetriebsstatus"
    ]].sort_values("Bruttoleistung", ascending=False).head(20))